# CrisisGraph ML-S1: serving Qwen3-4B + SFT-V4 on a Kaggle T4

Plan ML-S1 (approved 2026-11-02). Same fine-tuned model, served several ways on one T4:
which gives the best trade-off between **triage accuracy** (blind `triage-eval-v3.1.0`, 120 cases,
scored exactly as in the SFT-V4 report) and **latency and throughput**?

| Variant | What it is | Session |
|---|---|---|
| **B** | vLLM, FP16 base + SFT-V4 LoRA at run time | 1 |
| **C** | vLLM, FP16 with the adapter merged | 2 |
| **D** | vLLM, published 4-bit AWQ base + LoRA | 2 |
| **A** | llama.cpp CUDA, Q4_K_M + `adapter-f16.gguf` (the files validated locally) | 2 |

**Before running:** notebook settings → Accelerator **GPU T4** (one GPU is used), Internet **on**,
and attach the private dataset `crisisgraph-ml-s1` (made by `ml/scripts/package_kaggle_dataset.sh`).

**Privacy:** keep this notebook and the dataset private. The blind set is only sent to the model
served inside this notebook, never to a hosted API.

**Parity gates** (declared in the plan, against the local SFT-V4 result): 120/120 usable outputs,
critical under-triage ≤ 8 (local 7), V2 exact ≥ 78 (local 81), invented-fact cases ≤ 34 (local 31),
review recall ≥ 0.89 (local 0.91). A variant that misses a gate is reported as a failed trade-off.

**At the end of each session**, run the last cell and download `ml-s1-results.zip` from the output panel.

## 0. Settings

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

# Pinned versions. VLLM_VERSION is checked in step 1; if it does not run on the T4,
# change it here, rerun from this cell, and note the change in the report.
VLLM_VERSION = "0.11.0"
# vLLM 0.11 does not cap transformers, and transformers 5 removed a tokenizer attribute it uses
# (all_special_tokens_extended). Pin a 4.x release inside vLLM's range (>= 4.56).
TRANSFORMERS_VERSION = "4.57.1"
BASE_REPO = "Qwen/Qwen3-4B-Instruct-2507"
BASE_REV = "cdbee75f17c01a7cc42f958dc650907174af0554"      # same revision as SFT-V4 training
GGUF_REPO = "lmstudio-community/Qwen3-4B-Instruct-2507-GGUF"
GGUF_REV = "4edb920b6f14e3b9284d4502a6485103d72cde05"
GGUF_FILE = "Qwen3-4B-Instruct-2507-Q4_K_M.gguf"
GGUF_SHA256 = "8cdb57cbb880d313736a9bc4e3d3d2485f145b5e19cf33783746e753e82641fc"
LLAMA_CPP_TAG = "b11429"                                    # build used for the local SFT-V4 evaluation
LLAMA_CPP_COMMIT_PREFIX = "d812350493"
# Variant D: a published AWQ of the same base. Chosen in step 5 from the listed candidates.
AWQ_REPO = None
AWQ_REV = None

GATES = {"usable_outputs": (">=", 120), "critical_under_triage": ("<=", 8), "v2_exact_match": (">=", 78),
         "invented_fact_cases": ("<=", 34), "v2_review_recall": (">=", 0.89)}

WORK = Path("/kaggle/working")
RUNS = WORK / "runs"                 # results: kept and downloaded
SCRATCH = Path("/tmp/ml-s1")         # models, venv, builds: large, not kept (20 GB cap on /kaggle/working)
for p in (RUNS, SCRATCH):
    p.mkdir(parents=True, exist_ok=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4
# On the T4 (no FlashAttention), vLLM 0.11 picks FlexAttention: measured 9.1 tokens/s for the plain base
# model against 19.4 with Triton attention (session 1, 2026-11-03). FlashInfer is not installed with vLLM 0.11.
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

## 1a. Bundle: verify and install the evaluator

In [ ]:
found = glob.glob("/kaggle/input/**/BUNDLE.json", recursive=True)
assert found, "attach the private dataset crisisgraph-ml-s1 (Add input)"
SRC = Path(found[0]).parent
BUNDLE = SCRATCH / "bundle"
if not BUNDLE.exists():
    shutil.copytree(SRC, BUNDLE)
bad = subprocess.run(["sha256sum", "-c", "--quiet", "SHA256SUMS"], cwd=BUNDLE, capture_output=True, text=True)
assert bad.returncode == 0, bad.stdout + bad.stderr
print("bundle checksums ok:", json.loads((BUNDLE / "BUNDLE.json").read_text())["git_commit"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(BUNDLE / "ml")], check=True)

ML = BUNDLE / "ml"
DATASET = ML / "data" / "triage-eval-v3.1.0.jsonl"
PROMPT = ML / "prompts" / "triage-extraction-v3.2.txt"
SCHEMA = BUNDLE / "schemas" / "triage-extraction-v3.json"
LEAK_AUDIT = ML / "data" / "triage-sft-v4-leak-audit.json"
ADAPTER_PEFT = BUNDLE / "adapter" / "peft"
ADAPTER_GGUF = BUNDLE / "adapter" / "adapter-f16.gguf"
# The running kernel does not see an editable install made after it started (new processes do).
if str(ML / "src") not in sys.path:
    sys.path.insert(0, str(ML / "src"))
import crisisgraph_ml
print("crisisgraph_ml from", crisisgraph_ml.__file__)

## 1b. Environment record

In [ ]:
def sh(cmd, **kw):
    """Run a shell command, return stdout (raises on failure)."""
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True, **kw).stdout.strip()

env = {
    "recorded": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader"),
    "cuda_runtime_nvcc": sh("nvcc --version | tail -1 || true"),
    "python": sys.version,
    "cpu": sh("lscpu | grep 'Model name' || true"),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
    "attention_backend": os.environ.get("VLLM_ATTENTION_BACKEND"),
}
print(json.dumps(env, indent=2))

## 1c. Install the pinned vLLM in its own environment
Kaggle's own torch stays untouched; vLLM runs from `/tmp/ml-s1/venv-vllm`.

In [ ]:
VENV = SCRATCH / "venv-vllm"
VPY = VENV / "bin" / "python"
if not VPY.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    sh(f"uv venv --python 3.12 {VENV}")
# Always run (fast when already satisfied), so a changed pin also fixes an existing environment.
sh(f'uv pip install --python {VPY} "vllm=={VLLM_VERSION}" "transformers=={TRANSFORMERS_VERSION}" peft huggingface_hub')
env["vllm_stack"] = {k: sh(f"{VPY} -c 'import {k}; print({k}.__version__)'") for k in ("vllm", "torch", "transformers", "peft")}
env["torch_cuda"] = sh(f"{VPY} -c 'import torch; print(torch.version.cuda, torch.cuda.get_device_capability())'")
print(env["vllm_stack"], env["torch_cuda"])

## 1d. Base model at the pinned revision

In [ ]:
from huggingface_hub import snapshot_download
BASE_DIR = Path(snapshot_download(BASE_REPO, revision=BASE_REV, local_dir=SCRATCH / "base"))
print(BASE_DIR, sh(f"du -sh {BASE_DIR}"))

## Helpers: servers, evaluation, gates

In [ ]:
SERVER = {"proc": None}

def start_server(cmd, name, timeout=1200):
    """Start an OpenAI-compatible server and wait until it answers. Logs go to runs/<name>-server.log."""
    stop_server()
    log = open(RUNS / f"{name}-server.log", "w")
    log.write(" ".join(map(str, cmd)) + "\n"); log.flush()
    SERVER["proc"] = subprocess.Popen(list(map(str, cmd)), stdout=log, stderr=subprocess.STDOUT)
    SERVER["name"] = name
    started = time.time()
    while time.time() - started < timeout:
        if SERVER["proc"].poll() is not None:
            raise RuntimeError(f"server exited; see runs/{name}-server.log:\n" + tail(name))
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=5)
            print(f"{name}: ready after {time.time() - started:.0f} s")
            return
        except OSError:
            time.sleep(5)
    raise TimeoutError(f"{name} did not start; see runs/{name}-server.log")

def stop_server():
    proc = SERVER.get("proc")
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(60)
        except subprocess.TimeoutExpired:
            proc.kill()
    SERVER["proc"] = None
    time.sleep(5)

def tail(name, n=40):
    return "\n".join((RUNS / f"{name}-server.log").read_text(errors="replace").splitlines()[-n:])

def vllm_cmd(model, *extra, lora=True):
    cmd = [VENV / "bin" / "vllm", "serve", model, "--port", PORT, "--dtype", "half", "--max-model-len", 4096,
           "--gpu-memory-utilization", 0.90, "--seed", 0, "--generation-config", "vllm"]
    if lora:
        cmd += ["--enable-lora", "--lora-modules", f"sftv4={ADAPTER_PEFT}", "--max-lora-rank", 16]
    return cmd + list(extra)

def evaluate(name, backend, model, adapter="none"):
    """Blind v3.1.0, serially, as in the SFT-V4 report; then rescored with urgency-rules-v5 (all 120, and
    the 92 without a close training neighbour). Resumable: rerun to continue after an interruption."""
    out = RUNS / f"acc-{name}.json"
    subprocess.run([sys.executable, "-m", "crisisgraph_ml.triage_eval_v3", "--backend", backend,
                    "--base-url", BASE_URL, "--model", model, "--adapter", str(adapter), "--dataset", DATASET,
                    "--prompt", PROMPT, "--schema", SCHEMA, "--max-tokens", "768", "--output", out], check=True)
    for suffix, extra in (("rules-v5", []), ("rules-v5-no-neighbour", ["--exclude-from", LEAK_AUDIT])):
        subprocess.run([sys.executable, "-c", "from crisisgraph_ml.triage_eval_v3 import main_rescore; main_rescore()",
                        out, "--dataset", DATASET, "--rules", "urgency-rules-v5",
                        "--output", RUNS / f"acc-{name}-{suffix}.json", *extra], check=True)
    return gate_check(name)

def gate_check(name):
    from crisisgraph_ml.triage_eval_v3 import headline
    head = headline(json.loads((RUNS / f"acc-{name}-rules-v5.json").read_text()))
    value = lambda k: len(head[k]) if isinstance(head[k], list) else head[k]   # some measures are lists of case ids
    passed = {k: (value(k) >= v if op == ">=" else value(k) <= v) for k, (op, v) in GATES.items()}
    head["gates_passed"] = passed
    head["all_gates_passed"] = all(passed.values())
    (RUNS / f"acc-{name}-headline.json").write_text(json.dumps(head, indent=2))
    print(json.dumps(head, indent=2))
    return head

def bench(name, model, backend="vllm", concurrency="1,2,4,8,16,32", per_level=64, reference=None,
          limit=0, notes=""):
    """Speed only (streaming). Answers are fingerprinted; with reference=..., changed answers are listed."""
    cmd = [sys.executable, "-m", "crisisgraph_ml.serving_bench", "--base-url", BASE_URL, "--backend", backend,
           "--model", model, "--dataset", DATASET, "--prompt", PROMPT, "--schema", SCHEMA,
           "--concurrency", concurrency, "--requests-per-level", str(per_level), "--gpu-sample",
           "--label", name, "--server-notes", notes, "--raw", RUNS / f"bench-{name}.jsonl",
           "--output", RUNS / f"bench-{name}.json"]
    if limit:
        cmd += ["--limit", str(limit)]
    if reference:
        cmd += ["--reference", RUNS / f"bench-{reference}.json"]
    subprocess.run(list(map(str, cmd)), check=True)
    return json.loads((RUNS / f"bench-{name}.json").read_text())

## 1. Feasibility (session 1, about 30 minutes): vLLM FP16 + LoRA on the T4

One request with the strict JSON schema. If the server does not start, or the request fails,
**stop here**: note the error from the log, and try the newest vLLM version that supports compute
capability 7.5 (change `VLLM_VERSION` in step 0). Do not continue to step 2 on a broken setup.

In [ ]:
start_server(vllm_cmd(BASE_DIR), "B-feasibility")
from crisisgraph_ml.triage_eval import load_cases, request_payload
from jsonschema import Draft202012Validator
schema = json.loads(SCHEMA.read_text())
payload = request_payload("vllm", "sftv4", PROMPT.read_text().strip(), schema,
                          "Water is waist deep near Aluva town hall, 3 of us, grandmother cannot walk. I am here. Send an ambulance.", 768)
req = urllib.request.Request(f"{BASE_URL}/chat/completions", data=json.dumps(payload).encode(),
                             headers={"content-type": "application/json"})
t = time.time()
try:
    body = json.load(urllib.request.urlopen(req, timeout=300))
except urllib.error.HTTPError as e:
    print("HTTP", e.code, e.read().decode()[:2000]); raise
answer = json.loads(body["choices"][0]["message"]["content"])
Draft202012Validator(schema).validate(answer)
env["feasibility"] = {"ok": True, "latency_s": round(time.time() - t, 2), "usage": body.get("usage"),
                      "attention_backend": [l for l in tail("B-feasibility", 400).splitlines() if "backend" in l.lower()][:5]}
print(json.dumps(answer, indent=2, ensure_ascii=False)[:1500])
print(json.dumps(env["feasibility"], indent=2))
(RUNS / "environment.json").write_text(json.dumps(env, indent=2))

## 2. Variant B: accuracy (blind v3.1.0, serial, about 10–20 minutes)

In [ ]:
head_B = evaluate("B", "vllm", "sftv4")

## 3. Variant B: speed sweep (concurrency 1 to 32)
The concurrency-1 level is the reference for answer fingerprints.

In [ ]:
bench_B = bench("B", "sftv4", notes=" ".join(map(str, vllm_cmd(BASE_DIR))))

### 3b. Speed settings on B
Each one restarts the server. Answers are compared with the B run above (`answers_changed_vs_reference`):
a setting that changes answers is not speed-only and needs its own accuracy run.

In [ ]:
SPEED_SETTINGS = {
    # Session 1: the grammar alone took the base model from 19.4 to 10.9 tokens/s; compare the two libraries.
    "B-grammar-xgrammar": ["--structured-outputs-config", json.dumps({"backend": "xgrammar"})],
    "B-grammar-guidance": ["--structured-outputs-config", json.dumps({"backend": "guidance"})],
    "B-no-prefix-cache": ["--no-enable-prefix-caching"],
    "B-max-seqs-8": ["--max-num-seqs", 8],
    "B-ngram-spec": ["--speculative-config", json.dumps({"method": "ngram", "num_speculative_tokens": 4,
                                                         "prompt_lookup_max": 4, "prompt_lookup_min": 2})],
}
speed = {}
for name, extra in SPEED_SETTINGS.items():
    try:
        start_server(vllm_cmd(BASE_DIR, *extra), name)
        speed[name] = bench(name, "sftv4", concurrency="1,8,32", reference="B",
                            notes=" ".join(map(str, vllm_cmd(BASE_DIR, *extra))))
    except Exception as error:   # one failed setting must not stop the others; recorded
        speed[name] = {"error": str(error)[:2000]}
        print(name, "FAILED:", str(error)[:500])
stop_server()

### 3c. Parity at high concurrency
20 cases at concurrency 16 must give the same answers as the serial run.

In [ ]:
start_server(vllm_cmd(BASE_DIR), "B-parity")
parity = bench("B-parity-c16", "sftv4", concurrency="16", per_level=20, limit=20, reference="B")
print("changed answers at concurrency 16:", parity["levels"][0]["answers_changed_vs_reference"])
stop_server()

---
# Session 2

Rerun steps 0, 1a, 1b, 1c, 1d and the helpers first (the scratch disk is not kept between sessions).

## 4. Variant C: adapter merged into the FP16 weights

In [ ]:
MERGED = SCRATCH / "merged"
if not (MERGED / "config.json").exists():
    merge = f"""
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
base = AutoModelForCausalLM.from_pretrained("{BASE_DIR}", torch_dtype=torch.float16)
merged = PeftModel.from_pretrained(base, "{ADAPTER_PEFT}").merge_and_unload()
merged.save_pretrained("{MERGED}", safe_serialization=True)
AutoTokenizer.from_pretrained("{BASE_DIR}").save_pretrained("{MERGED}")
print("merged")
"""
    subprocess.run([str(VPY), "-c", merge], check=True)
start_server(vllm_cmd(MERGED, "--served-model-name", "sftv4-merged", lora=False), "C")
head_C = evaluate("C", "vllm", "sftv4-merged")
bench_C = bench("C", "sftv4-merged", notes=" ".join(map(str, vllm_cmd(MERGED, lora=False))))
stop_server()

## 5. Variant D: published 4-bit AWQ base + LoRA

The plan uses a **published** AWQ of the same base. The cell lists candidates on Hugging Face.
Pick one whose model card states it is an AWQ of `Qwen3-4B-Instruct-2507`, set `AWQ_REPO` and
`AWQ_REV` (a commit hash from the repo's history) in step 0, and rerun from there.

In [ ]:
from huggingface_hub import HfApi
for m in HfApi().list_models(search="Qwen3-4B-Instruct-2507 AWQ", sort="downloads", limit=15):
    print(m.id, getattr(m, "downloads", None))

In [ ]:
assert AWQ_REPO and AWQ_REV, "set AWQ_REPO and AWQ_REV in step 0"
AWQ_DIR = Path(snapshot_download(AWQ_REPO, revision=AWQ_REV, local_dir=SCRATCH / "awq"))
start_server(vllm_cmd(AWQ_DIR), "D")
head_D = evaluate("D", "vllm", "sftv4")
bench_D = bench("D", "sftv4", notes=f"{AWQ_REPO}@{AWQ_REV} " + " ".join(map(str, vllm_cmd(AWQ_DIR))))
stop_server()

## 6. Variant A: llama.cpp CUDA reference (the locally validated files)

Built from tag `b11429`, the build of the local SFT-V4 evaluation, with CUDA for the T4 (about 10–15 minutes).
One slot, as locally; speed at concurrency 1 only (it is the quality reference, not a throughput competitor).

In [ ]:
LLAMA = SCRATCH / "llama.cpp"
SERVER_BIN = LLAMA / "build" / "bin" / "llama-server"
if not SERVER_BIN.exists():
    sh(f"git clone --depth 1 --branch {LLAMA_CPP_TAG} https://github.com/ggml-org/llama.cpp {LLAMA}")
    commit = sh(f"git -C {LLAMA} rev-parse HEAD")
    assert commit.startswith(LLAMA_CPP_COMMIT_PREFIX), commit
    sh(f"cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF")
    sh(f"cmake --build {LLAMA}/build --target llama-server -j {os.cpu_count()}")
from huggingface_hub import hf_hub_download
GGUF = Path(hf_hub_download(GGUF_REPO, GGUF_FILE, revision=GGUF_REV, local_dir=SCRATCH / "gguf"))
assert hashlib.sha256(GGUF.read_bytes()).hexdigest() == GGUF_SHA256
env["llama_cpp"] = {"tag": LLAMA_CPP_TAG, "commit": sh(f"git -C {LLAMA} rev-parse HEAD")}
start_server([SERVER_BIN, "--model", GGUF, "--lora", ADAPTER_GGUF, "--alias", "local", "--host", "127.0.0.1",
              "--port", PORT, "--ctx-size", 4096, "--parallel", 1, "-ngl", 99], "A")
head_A = evaluate("A", "llama-cpp", "local", adapter="adapter-f16.gguf")
bench_A = bench("A", "local", backend="llama-cpp", concurrency="1", per_level=32)
stop_server()

## 7. Summary of this session and download

In [ ]:
summary = {"environment": env, "variants": {}}
for name in ("A", "B", "C", "D"):
    entry = {}
    h = RUNS / f"acc-{name}-headline.json"
    if h.exists():
        entry["accuracy"] = json.loads(h.read_text())
    b = RUNS / f"bench-{name}.json"
    if b.exists():
        entry["speed"] = {lvl["concurrency"]: {k: lvl[k] for k in ("requests_per_s", "output_tokens_per_s", "failed")}
                          | {"e2e_p50": lvl["e2e_s"]["p50"], "e2e_p95": lvl["e2e_s"]["p95"],
                             "ttft_p50": lvl["ttft_s"]["p50"], "gpu_mib": lvl.get("gpu_memory_peak_mib")}
                          for lvl in json.loads(b.read_text())["levels"]}
    if entry:
        summary["variants"][name] = entry
(RUNS / "environment.json").write_text(json.dumps(env, indent=2))
(RUNS / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps({n: {"gates": v.get("accuracy", {}).get("all_gates_passed"),
                      "c1_e2e_p50": v.get("speed", {}).get(1, {}).get("e2e_p50"),
                      "best_req_s": max((s["requests_per_s"] or 0) for s in v["speed"].values()) if "speed" in v else None}
                  for n, v in summary["variants"].items()}, indent=2))
stop_server()
shutil.make_archive(str(WORK / "ml-s1-results"), "zip", RUNS)
print("download /kaggle/working/ml-s1-results.zip")